# 🗄️ Phase 3: SQL Analysis
## Credit Risk Analytics — Home Credit Default Risk
---
**Goal:** Load cleaned data into SQLite, run analytical SQL queries to produce default rate segmentations used in the Power BI dashboard.


In [1]:
import pandas as pd
import numpy as np
import sqlite3
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv('application_clean.csv')
print(f"Dataset loaded: {df.shape}")


Dataset loaded: (7659, 110)


## 1. Load Data into SQLite

In [2]:
# Create SQLite DB (equivalent to MySQL locally)
conn = sqlite3.connect('credit_risk.db')

# Load main table
df.to_sql('applications', conn, if_exists='replace', index=False)
print("Table 'applications' created in SQLite ✅")

# Verify
result = pd.read_sql("SELECT COUNT(*) as total_rows FROM applications", conn)
print(result)


Table 'applications' created in SQLite ✅
   total_rows
0        7659


## 2. Query 1 — Default Rate by Income Bracket

In [3]:
query1 = '''
SELECT
    INCOME_BRACKET,
    COUNT(*) AS applicants,
    ROUND(SUM(TARGET) * 100.0 / COUNT(*), 2) AS default_rate_pct,
    ROUND(AVG(AMT_INCOME_TOTAL), 0) AS avg_income
FROM applications
WHERE INCOME_BRACKET != ''
GROUP BY INCOME_BRACKET
ORDER BY default_rate_pct DESC
'''

default_by_income = pd.read_sql(query1, conn)
print("=== Default Rate by Income Bracket ===")
print(default_by_income.to_string(index=False))


=== Default Rate by Income Bracket ===
INCOME_BRACKET  applicants  default_rate_pct  avg_income
      90K-180K        3874              8.34   126045.00
     180K-270K        1914              8.15   206207.00
          <90K         912              6.91    67135.00
         400K+         199              6.53   445379.00
     270K-400K         759              5.01   308140.00


## 3. Query 2 — Default Rate by Age Group

In [4]:
query2 = '''
SELECT
    AGE_GROUP,
    COUNT(*) AS applicants,
    ROUND(SUM(TARGET) * 100.0 / COUNT(*), 2) AS default_rate_pct
FROM applications
GROUP BY AGE_GROUP
ORDER BY default_rate_pct DESC
'''

default_by_age = pd.read_sql(query2, conn)
print("=== Default Rate by Age Group ===")
print(default_by_age.to_string(index=False))


=== Default Rate by Age Group ===
AGE_GROUP  applicants  default_rate_pct
      <25         318             12.26
    25-35        1791             10.94
    35-45        2062              7.42
    45-55        1824              6.58
      55+        1663              5.11
     None           1              0.00


## 4. Query 3 — Portfolio Exposure by Loan Type

In [5]:
query3 = '''
SELECT
    NAME_CONTRACT_TYPE,
    COUNT(*) AS total_loans,
    ROUND(SUM(AMT_CREDIT) / 1000000.0, 2) AS total_credit_millions,
    ROUND(AVG(AMT_CREDIT), 0) AS avg_credit
FROM applications
GROUP BY NAME_CONTRACT_TYPE
'''

portfolio_exposure = pd.read_sql(query3, conn)
print("=== Portfolio Exposure by Loan Type ===")
print(portfolio_exposure.to_string(index=False))


=== Portfolio Exposure by Loan Type ===
NAME_CONTRACT_TYPE  total_loans  total_credit_millions  avg_credit
        Cash loans         6893                4338.87   629460.00
   Revolving loans          766                 235.80   307833.00


## 5. Query 4 — High Risk Applicants

In [6]:
query4 = '''
SELECT
    SK_ID_CURR,
    NAME_CONTRACT_TYPE,
    AMT_CREDIT,
    AMT_INCOME_TOTAL,
    AGE,
    AGE_GROUP,
    INCOME_BRACKET,
    ANNUITY_INCOME_RATIO,
    TARGET
FROM applications
WHERE TARGET = 1
ORDER BY AMT_CREDIT DESC
LIMIT 20
'''

high_risk = pd.read_sql(query4, conn)
print("=== Top 20 High-Risk Applicants ===")
print(high_risk.to_string(index=False))


=== Top 20 High-Risk Applicants ===
 SK_ID_CURR NAME_CONTRACT_TYPE  AMT_CREDIT  AMT_INCOME_TOTAL   AGE AGE_GROUP INCOME_BRACKET  ANNUITY_INCOME_RATIO  TARGET
     100784         Cash loans  1826764.20          54000.00 60.00       55+           <90K                  1.37       1
     105925         Cash loans  1826764.20         247500.00 50.00     45-55      180K-270K                  0.28       1
     100273         Cash loans  1710000.00         157500.00 63.00       55+       90K-180K                  0.42       1
     104209         Cash loans  1575000.00         270000.00 30.00     25-35      270K-400K                  0.16       1
     104917         Cash loans  1575000.00         202500.00 37.00     35-45      180K-270K                  0.21       1
     107693         Cash loans  1575000.00         193500.00 39.00     35-45      180K-270K                  0.22       1
     106052         Cash loans  1546020.00         180000.00 57.00       55+      180K-270K                  0

## 6. Query 5 — Default Rate by Loan Type & Age Group (Cross-Analysis)

In [7]:
query5 = '''
SELECT
    NAME_CONTRACT_TYPE,
    AGE_GROUP,
    COUNT(*) AS applicants,
    ROUND(SUM(TARGET) * 100.0 / COUNT(*), 2) AS default_rate_pct
FROM applications
GROUP BY NAME_CONTRACT_TYPE, AGE_GROUP
ORDER BY default_rate_pct DESC
'''

cross_analysis = pd.read_sql(query5, conn)
print("=== Cross Analysis: Loan Type × Age Group ===")
print(cross_analysis.to_string(index=False))


=== Cross Analysis: Loan Type × Age Group ===
NAME_CONTRACT_TYPE AGE_GROUP  applicants  default_rate_pct
        Cash loans       <25         232             15.52
        Cash loans     25-35        1577             11.35
   Revolving loans     25-35         214              7.94
        Cash loans     35-45        1872              7.80
        Cash loans     45-55        1668              6.89
        Cash loans       55+        1543              5.31
   Revolving loans     35-45         190              3.68
   Revolving loans       <25          86              3.49
   Revolving loans     45-55         156              3.21
   Revolving loans       55+         120              2.50
        Cash loans      None           1              0.00


## 7. Export Results

In [8]:
# Export all query results for Power BI / reporting
default_by_income.to_csv('sql_default_by_income.csv', index=False)
default_by_age.to_csv('sql_default_by_age.csv', index=False)
portfolio_exposure.to_csv('sql_portfolio_exposure.csv', index=False)
high_risk.to_csv('sql_high_risk_applicants.csv', index=False)

print("All SQL results exported ✅")
print("Files: sql_default_by_income.csv, sql_default_by_age.csv,")
print("       sql_portfolio_exposure.csv, sql_high_risk_applicants.csv")

conn.close()


All SQL results exported ✅
Files: sql_default_by_income.csv, sql_default_by_age.csv,
       sql_portfolio_exposure.csv, sql_high_risk_applicants.csv
